# FANNS microbench (Colab) — synthetic PRE/POST

**Purpose:** Run a small filtered-ANN microbench on Google Colab and export artifacts for the `rag-vector-query-lang` repo.

**Not P0 by default:** This notebook uses **synthetic** vectors first so it runs without huge downloads. That is useful for plumbing. It is **not** a substitute for a licensed SIFT1M (or equivalent) protocol-01 P0 run.

**Do not** paste ACORN / paper table numbers into `metrics.json`.

**Return path:** zip → download → unzip into `experiments/results/fanns/<run_id>/` → commit metrics + ENV + plans (see `experiments/HOW_TO_PROVIDE_RESULTS.md`).


## 1. Install ANN library

Prefer **faiss-cpu** (works on Colab CPU and is the reliable default). If the runtime has a GPU, try `faiss-gpu` and fall back to CPU on failure. `hnswlib` is an optional alternative if FAISS install fails.


In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *pkgs])

# Detect GPU (optional)
has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    # torch may be absent; check nvidia-smi lightly
    try:
        subprocess.check_call(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        has_gpu = True
    except Exception:
        has_gpu = False

backend = None
err = None
if has_gpu:
    try:
        pip_install("faiss-gpu")
        import faiss
        backend = f"faiss-gpu ({faiss.__version__ if hasattr(faiss, '__version__') else 'ok'})"
    except Exception as e:
        err = e
        print("faiss-gpu failed, falling back to faiss-cpu:", e)

if backend is None:
    try:
        pip_install("faiss-cpu")
        import faiss
        backend = f"faiss-cpu ({getattr(faiss, '__version__', 'ok')})"
    except Exception as e:
        err = e
        print("faiss-cpu failed, trying hnswlib:", e)
        pip_install("hnswlib")
        import hnswlib
        backend = "hnswlib"
        faiss = None

pip_install("numpy")
import numpy as np

print("ANN backend:", backend)
print("numpy:", np.__version__)
print("GPU runtime detected:", has_gpu)


## 2. Config + output folder

Synthetic defaults: N in [50k, 100k], d=128, selectivities {0.01, 0.05, 0.1, 0.5}.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, platform, os, time, statistics, zipfile

# --- knobs (safe for Colab free tier; raise N toward 1e5 if RAM allows) ---
N = 50_000          # try 100_000 if the runtime has enough RAM
D = 128
N_QUERIES = 100
K = 10
SEED = 42
SELECTIVITIES = [0.01, 0.05, 0.1, 0.5]
POST_CAND_MULT = 50  # POST: retrieve k * mult then filter

run_id = "colab_synth_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
OUT = Path(f"/content/rql_fanns_{run_id}")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "plans").mkdir(exist_ok=True)
print("OUT =", OUT)
print("run_id =", run_id)


## 3. Synthetic data + predicates

Gaussian vectors, L2-normalized. Independent random predicates at each selectivity (fixed seed).


In [ ]:
rng = np.random.default_rng(SEED)

def l2_normalize(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-12)

db = l2_normalize(rng.standard_normal((N, D), dtype=np.float64).astype(np.float32))
queries = l2_normalize(rng.standard_normal((N_QUERIES, D), dtype=np.float64).astype(np.float32))
base_labels = rng.random(N)  # U[0,1) for thresholding selectivity

print(db.shape, queries.shape, "bytes≈", db.nbytes // (1024 * 1024), "MiB")


## 4. Index helpers — PRE / POST

- **POST** = ANN over full DB (oversized candidate list) then filter to predicate.
- **PRE** = filter ids, then search (HNSW/Flat on subset, or brute on subset if small).

Recall@10 is measured against **filtered** brute-force ground truth (same predicate).


In [ ]:
def brute_topk(db_vecs, q_vecs, k):
    sims = q_vecs @ db_vecs.T
    k = min(k, db_vecs.shape[0])
    part = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    rows = np.arange(q_vecs.shape[0])[:, None]
    order = np.argsort(-sims[rows, part], axis=1)
    return part[rows, order]


def build_index(vecs):
    """Return (kind, handle). kind in {faiss, hnsw, brute}."""
    if 'faiss' in globals() and faiss is not None:
        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs.astype(np.float32))
        return "faiss", index
    try:
        import hnswlib
        index = hnswlib.Index(space="ip", dim=vecs.shape[1])
        index.init_index(max_elements=vecs.shape[0], ef_construction=100, M=16)
        index.add_items(vecs.astype(np.float32))
        index.set_ef(64)
        return "hnsw", index
    except Exception:
        return "brute", vecs


def search_index(kind, handle, q, k):
    if kind == "faiss":
        ntotal = handle.ntotal
    elif kind == "hnsw":
        ntotal = handle.get_current_count()
    else:
        ntotal = handle.shape[0]
    k = max(1, min(k, int(ntotal)))
    if kind == "faiss":
        _, I = handle.search(q.astype(np.float32), k)
        return I
    if kind == "hnsw":
        labels, _ = handle.knn_query(q.astype(np.float32), k=k)
        return labels.astype(np.int64)
    return brute_topk(handle, q, k)


def filtered_gt(db_vecs, q_vecs, mask, k):
    ids = np.flatnonzero(mask)
    if ids.size == 0:
        return np.full((q_vecs.shape[0], k), -1, dtype=np.int64)
    local = brute_topk(db_vecs[ids], q_vecs, min(k, ids.size))
    out = np.full((q_vecs.shape[0], k), -1, dtype=np.int64)
    mapped = ids[local]
    out[:, : mapped.shape[1]] = mapped
    return out


def recall_at_k(pred, truth):
    nq, k = pred.shape
    hits = 0
    for i in range(nq):
        hits += len(set(pred[i].tolist()) & set(truth[i].tolist()))
    return hits / (nq * k)


def percentile(xs, p):
    return float(np.percentile(np.asarray(xs, dtype=np.float64), p)) if xs else 0.0


# Full-DB index for POST
full_kind, full_index = build_index(db)
print("Full index:", full_kind)


In [ ]:
def run_post(mask, k=K, cand_mult=POST_CAND_MULT):
    n = db.shape[0]
    cand_k = min(n, max(k * cand_mult, k))
    times, preds = [], np.full((N_QUERIES, k), -1, dtype=np.int64)
    for i in range(N_QUERIES):
        t0 = time.perf_counter()
        cands = search_index(full_kind, full_index, queries[i:i+1], cand_k)[0]
        kept = [int(c) for c in cands if c >= 0 and mask[c]]
        if len(kept) < k:
            ids = np.flatnonzero(mask)
            if ids.size:
                local = brute_topk(db[ids], queries[i:i+1], min(k, ids.size))[0]
                for j in ids[local]:
                    if int(j) not in kept:
                        kept.append(int(j))
                    if len(kept) >= k:
                        break
        preds[i, :min(k, len(kept))] = kept[:k]
        times.append((time.perf_counter() - t0) * 1000.0)
    return preds, times


def run_pre(mask, k=K):
    ids = np.flatnonzero(mask)
    times, preds = [], np.full((N_QUERIES, k), -1, dtype=np.int64)
    if ids.size == 0:
        return preds, [0.0] * N_QUERIES
    sub = db[ids]
    # For modest subsets, Flat/HNSW on subset; for tiny, brute is fine
    kind, handle = build_index(sub)
    for i in range(N_QUERIES):
        t0 = time.perf_counter()
        local = search_index(kind, handle, queries[i:i+1], min(k, ids.size))[0]
        preds[i, :local.shape[0]] = ids[local]
        times.append((time.perf_counter() - t0) * 1000.0)
    return preds, times


## 5. Sweep selectivities × modes → metrics


In [ ]:
rows = []
for s in SELECTIVITIES:
    mask = base_labels < s
    if mask.sum() < K:
        need = K - int(mask.sum())
        zeros = np.flatnonzero(~mask)[:need]
        mask = mask.copy()
        mask[zeros] = True
    gt = filtered_gt(db, queries, mask, K)
    print(f"selectivity={s} matched={int(mask.sum())}")

    for mode, runner in (("PRE", run_pre), ("POST", run_post)):
        pred, times = runner(mask)
        rec = recall_at_k(pred, gt)
        mean_ms = statistics.mean(times) if times else 0.0
        row = {
            "selectivity": s,
            "mode": mode,
            "recall_at_10": round(float(rec), 6),
            "latency_p50_ms": round(percentile(times, 50), 4),
            "latency_p95_ms": round(percentile(times, 95), 4),
            "qps": round((1000.0 / mean_ms) if mean_ms > 0 else 0.0, 4),
            "n_queries": N_QUERIES,
        }
        rows.append(row)
        print(" ", row)

metrics = {
    "run_id": run_id,
    "label": "colab_synthetic — not P0 unless you also ran licensed large-set cells and recorded that clearly",
    "dataset": f"synthetic_gaussian N={N} d={D} seed={SEED}",
    "rows": rows,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
with (OUT / "metrics.jsonl").open("w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print("Wrote", OUT / "metrics.json")


## 6. Write `ENV.txt`, plans, README


In [ ]:
env_txt = f"""run_id={run_id}
label=colab_synthetic (not automatically P0)
ann_backend={backend}
numpy={np.__version__}
python={sys.version.split()[0]}
platform={platform.platform()}
gpu_runtime_detected={has_gpu}
n={N} d={D} n_queries={N_QUERIES} k={K} seed={SEED}
selectivities={SELECTIVITIES}
modes=PRE,POST
post_candidate_mult={POST_CAND_MULT}
colab=true
"""
(OUT / "ENV.txt").write_text(env_txt)

plan = {
    "notebook": "experiments/colab/fanns_microbench_colab.ipynb",
    "seed": SEED,
    "n": N,
    "d": D,
    "n_queries": N_QUERIES,
    "k": K,
    "selectivities": SELECTIVITIES,
    "modes": ["PRE", "POST"],
    "post_candidate_mult": POST_CAND_MULT,
    "ann_backend": backend,
}
(OUT / "plans" / "run.json").write_text(json.dumps(plan, indent=2) + "\n")

(OUT / "README.md").write_text(
    f"""# {run_id}

Colab FANNS microbench export (synthetic vectors by default).

- Place this folder at `experiments/results/fanns/{run_id}/` in the GitHub repo.
- Commit `metrics.json`, `ENV.txt`, `plans/` — not multi-GB binaries.
- Synthetic-only runs are **not** a substitute for SIFT1M P0.
- Do not paste paper/ACORN numbers here.

See `experiments/HOW_TO_PROVIDE_RESULTS.md`.
"""
)
print(list(OUT.rglob("*")))


## 7. (Optional) SIFT1M download — license required

**Only run if you accept the dataset license / terms for the mirror you use.**  
Raw vectors must **not** be committed to git; record digests in `experiments/datasets/README.md`.

This cell is a stub: fill in a mirror URL you are allowed to use, download outside git, and adapt paths. Leaving it unrun is fine for synthetic plumbing.


In [ ]:
# OPTIONAL — do not run blindly. Accept license first.
# Example sketch only (URLs change; verify yourself):
#
# SIFT1M_DIR = Path("/content/sift1M")
# SIFT1M_DIR.mkdir(exist_ok=True)
# # wget/curl your licensed mirror of sift/sift_base.fvecs etc. into SIFT1M_DIR
# # Then rebuild db/queries from fvecs and re-run the sweep cells with a new run_id
# # labeled clearly as sift1m_* in metrics.json / ENV.txt.
#
print("SIFT1M cell skipped by default. Synthetic metrics already written under", OUT)
print("For P0: use a licensed set + document digests in experiments/datasets/README.md")


## 8. Zip + download instructions

Download the zip from Colab’s file browser (or the link this cell prints). Then on your machine:

```bash
cd /path/to/rag-vector-query-lang
unzip ~/Downloads/rql_fanns_<run_id>.zip -d experiments/results/fanns/
# expect: experiments/results/fanns/<run_id>/{metrics.json,ENV.txt,plans/,README.md}
git add experiments/results/fanns/<run_id>
git commit -m "results: FANNS Colab run <run_id>"
git push
```

Agents do not push. You do.


In [ ]:
zip_path = Path(f"/content/rql_fanns_{run_id}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            # archive as <run_id>/...
            arc = Path(run_id) / p.relative_to(OUT)
            zf.write(p, arcname=str(arc))

print("ZIP:", zip_path)
print("Size bytes:", zip_path.stat().st_size)
print()
print("Download this zip, then unzip into experiments/results/fanns/ in the repo.")
print("Target layout: experiments/results/fanns/%s/{metrics.json,ENV.txt,plans/,README.md}" % run_id)

# Colab helper (ignored outside Colab)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("(files.download unavailable — use the file browser to download)", e)
